In [ ]:
adjacency = {
    'A' : ['B', 'G', 'F'],
    'B' : ['A', 'C'],
    'C' : ['B', 'D', 'E', 'F'],
    'D' : ['C'],
    'E' : ['C'],
    'F' : ['A','C','G'],
    'G' : ['A', 'F']
}

colors = ['red', 'green', 'blue', 'yellow']

class DFSAgent:
    def __init__(self, colors, adjacency):
        self.colors=colors
        self.adjacency=adjacency
        self.state={} #state = {nodo:colore, 'A' : 'green', 'B' : None}

    def valid_coloring(self, state, new_node=None):
        # validare che i nodi in esame non violi i vincoli
        # nel caso in cui new_node non sia specificato cicliamo in tutti i nodi già con 
        # un'assegnazione parziale
        nodes_to_check = state.keys()
        if new_node is not None: 
            nodes_to_check = [new_node]

        for node in nodes_to_check:
            for neighbor in self.adjacency[node]:
                if state[node] == state.get(neighbor, None):
                    return False
        return True 

    # funzione successore dell'algoritmo di ricerca, 
    # genera i passi successivi validi 
    # genera gli stati in modo implicito, cioè solo quando richiesto 

    def next_states(self, state):
        for node in self.adjacency:
            if node not in state:
                for c in self.colors:
                    new_state = {**state, node : c}
                    if self.valid_coloring(new_state, new_node=None):
                        yield new_state
                break

    def dfs(self, state=None):
        if state==None:
            state={}
        if len(state)==len(self.adjacency):
            yield state
        for next_state in self.next_states(state):
            yield from self.dfs(next_state)

    


In [ ]:
class DFSAgentOptim(DFSAgent):
    def __init__(self, colors, adjacency, mrw=True,lcv=True,fwc=True):
        super().__init__(colors, adjacency)
        self.mrw, self.lcv, self.fwc = mrw,lcv,fwc

    def legal_values(self, state, node):
        return len(self.colors) - len({state[neighbor] for neighbor in self.adjacency[node]
                                       if neighbor in state})

    def forward_checking(self,state):
        # restiuiamo True solo se TUTTE le variabili
        # non assegnate hanno almeno un valore assegnabile
        return all(self.legal_values(state, node) > 0 for node 
                   in self.adjacency if node not in state) if self.fwc else True
    
    def MCV(self, state):
        return min([n for n in self.adjacency if n not in state], key=lambda n: self.legal_values(state,n) 
                   if self.mrw else 0)

    def removed_neighbors_colors(self, state, node, color):
         # LEAST CONSTRAINING VALUE, ci serve una funzione di utilità che conta quante opzioni
         # rimuove l'assegnazione di un valore e di una variabile 
        neighbors = [n for n in self.adjacency[node] if n not in state]
        return sum(self.legal_values(state,n) for n in neighbors) - sum(
            self.legal_values({**state, node : color}, n) for n in neighbors)

    # Fatto ciò possiamo usare il metodo sorted per ordinare i colori in ordine crescente di quanti
    # valori rimuovono se assegnati al nodo dato

    def LCV_sort(self,state,node):
        return sorted(self.colors, key=lambda c: 
                      self.removed_neighbors_colors(state, node, c) if self.lcv else 0)

    def next_states(self, state):
        node = self.MCV(state) # prima ottimizzazione fissa
        return (new_state for color in self.LCV_sort(state, node)
                if self.valid_coloring(new_state :={**state, node: color},
                    new_node=node) and self.forward_checking(new_state) 
        )

        #il fatto che return abbia un'espressione tra parentesi si chiama Generatore Lazy.
        #Ovvero i successori validi vengono generati uno alla volta solo quando richiesti 
        #dall'algoritmo di ricerca, senza occupare spazio in memoria.

        
